In [ ]:
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
FILE_NAMES = ['01-inventory.csv', '02-recipe.csv', '03-orders.csv', '04-git.csv']
REQUIRED_STRUCTURE = {
    '01-inventory.csv': ['idx', 'rm_name', 'stock_qty', 'unit', 'stock_price', 'eco_score'],
    '02-recipe.csv':    ['idx', 'recipe_number', 'product_name', 'rm_name', 'usage_percent'],
    '03-orders.csv': ['idx', 'mo_number', 'product_name', 'mfg_date', 'due_date', 'order_qty'],
    '04-git.csv':        ['idx', 'git_number', 'rm_name', 'git_qty', 'delivery_date']
}

## ---------------------------------------------------------
## SESSION 1: INITIAL FILE CHECK & DATA LOADING
## วัตถุประสงค์: ตรวจสอบความมีอยู่ของไฟล์และโครงสร้างเบื้องต้น
## ---------------------------------------------------------
print("--- SESSION 1: Initial File Check ---")
loaded_data = {}

for filename in FILE_NAMES:
    try:
        df = pd.read_csv(filename)
        df.columns = df.columns.str.strip() # ลบ space ที่ชื่อ column

        # จัดการชื่อ column พิเศษ
        if filename == '01-inventory.csv' and 'Unnamed: 6' in df.columns:
            df = df.rename(columns={'Unnamed: 6': 'total_stock_cost'})

        loaded_data[filename] = df
        print(f"✅ Loaded {filename} | Rows: {len(df)}")
    except FileNotFoundError:
        print(f"❌ Error: File {filename} not found.")

## ---------------------------------------------------------
## SESSION 2: DATA CLEANING & REFORMATTING
## วัตถุประสงค์: ล้างข้อมูลขยะ, แปลงตัวเลข และจัดการ Format วันที่
## ---------------------------------------------------------
print("\n--- SESSION 2: Data Cleaning & Reformatting ---")

for filename, df in loaded_data.items():
    # 1. ลบแถวว่าง (อิงจากคอลัมน์ idx)
    if 'idx' in df.columns:
        df = df.dropna(subset=['idx']).copy()

    # 2. ลบช่องว่างในข้อมูล String
    for col in df.select_dtypes(include=['object']):
        df[col] = df[col].astype(str).str.strip()

    # 3. แปลงค่าตัวเลข (ลบ comma)
    numeric_cols = ['stock_qty', 'stock_price', 'usage_percent', 'order_qty', 'git_qty']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace({',': ''}, regex=True), errors='coerce').fillna(0)

    # 4. แปลง Format วันที่ (YYYYMMDD)
    date_cols = ['mfg_date', 'due_date', 'delivery_date']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%Y%m%d', errors='coerce')

    loaded_data[filename] = df
    print(f"✨ Cleaned {filename}")

## ---------------------------------------------------------
## SESSION 3: MASTER DATA CONSISTENCY & RECIPE SUMMARY
## วัตถุประสงค์: ตรวจสอบความสอดคล้องของ RM และคำนวณต้นทุนสูตรผลิต
## ---------------------------------------------------------
print("\n--- SESSION 3: Recipe & Inventory Analysis ---")

df_inv = loaded_data['01-inventory.csv']
df_recipe = loaded_data['02-recipe.csv']

# ตรวจสอบ RM ที่มีในสูตรแต่ไม่มีในคลัง
recipe_rms = set(df_recipe['rm_name'].unique())
inv_rms = set(df_inv['rm_name'].unique())
missing_in_inv = recipe_rms - inv_rms

if missing_in_inv:
    print(f"⚠️ Warning: {len(missing_in_inv)} RM(s) missing in inventory. Auto-adding...")
    new_items = pd.DataFrame({
        'idx': range(len(df_inv)+1, len(df_inv)+1+len(missing_in_inv)),
        'rm_name': list(missing_in_inv),
        'stock_qty': 0, 'unit': 'ton', 'stock_price': 0, 'eco_score': 0
    })
    df_inv = pd.concat([df_inv, new_items], ignore_index=True)
    loaded_data['01-inventory.csv'] = df_inv

# คำนวณ Recipe Summary (ต้นทุนต่อหน่วย)
df_calc = pd.merge(df_recipe, df_inv[['rm_name', 'stock_price']], on='rm_name', how='left')
df_calc['line_cost'] = df_calc['stock_price'] * df_calc['usage_percent']

recipe_summary = df_calc.groupby(['recipe_number', 'product_name']).agg(
    total_usage_pct=('usage_percent', 'sum'),
    total_cost=('line_cost', 'sum'),
    items_count=('rm_name', 'count')
).reset_index()

recipe_summary.to_csv('05-recipe-summary.csv', index=False)
print(f"💾 Exported: 05-recipe-summary.csv (Rows: {len(recipe_summary)})")

## ---------------------------------------------------------
## SESSION 4: TIME BUCKET PROCESSING
## วัตถุประสงค์: แบ่งกลุ่มข้อมูลตามช่วงเวลา (7 วัน) สำหรับ Process ถัดไป
## ---------------------------------------------------------
print("\n--- SESSION 4: Time Bucket Processing ---")

df_orders = loaded_data['03-orders.csv']
df_git = loaded_data['04-git.csv']

# หาจุดเริ่มต้นของเวลา (Global Start Date)
global_start_date = min(df_orders['mfg_date'].min(), df_git['delivery_date'].min())
print(f"📍 Timeline Starts From: {global_start_date.date()}")

def apply_bucket(df, date_col):
    df = df.copy()
    # คำนวณ Bucket (7 วัน)
    df['bucket'] = ((df[date_col] - global_start_date).dt.days // 7) + 1
    # แปลงวันที่กลับเป็น String Format เพื่อ Export
    df[date_col] = df[date_col].dt.strftime('%Y%m%d')
    return df

df_orders_final = apply_bucket(df_orders, 'mfg_date')
df_git_final = apply_bucket(df_git, 'delivery_date')

# Export Result
df_orders_final.to_csv('06-orders-time-bucket.csv', index=False)
df_git_final.to_csv('07-git-time-buckets.csv', index=False)

print("✅ SESSION 4 Completed: All files are ready for next process.")
print("-" * 50)
print("Preview - Orders with Buckets:")
print(df_orders_final[['mo_number', 'mfg_date', 'bucket']].head())

--- SESSION 1: Initial File Check ---
✅ Loaded 01-inventory.csv | Rows: 20
✅ Loaded 02-recipe.csv | Rows: 38
✅ Loaded 03-orders.csv | Rows: 20
✅ Loaded 04-git.csv | Rows: 20

--- SESSION 2: Data Cleaning & Reformatting ---
✨ Cleaned 01-inventory.csv
✨ Cleaned 02-recipe.csv
✨ Cleaned 03-orders.csv
✨ Cleaned 04-git.csv

--- SESSION 3: Recipe & Inventory Analysis ---
💾 Exported: 05-recipe-summary.csv (Rows: 10)

--- SESSION 4: Time Bucket Processing ---
📍 Timeline Starts From: 2026-04-04
✅ SESSION 4 Completed: All files are ready for next process.
--------------------------------------------------
Preview - Orders with Buckets:
  mo_number  mfg_date  bucket
0     MO001  20260629      13
1     MO002  20260410       1
2     MO003  20260407       1
3     MO004  20260524       8
4     MO005  20260411       2
